# **TikTok Project**
**Course 1 — Foundations of Data Science**

**Author:** Ahmad Daniel  
**Date:** May 2026

---

## Project Overview

You have just started as a data professional at TikTok. The team is in the early stages of a claims classification project. TikTok's data must be examined to begin the process of Exploratory Data Analysis (EDA) in preparation for building a claims classification machine learning model.

**The purpose** of this project is to investigate and understand the data provided. This activity will:
1. Acquaint you with the data
2. Compile summary information about the data
3. Begin the process of EDA and reveal insights contained in the data
4. Prepare you for more in-depth EDA, hypothesis testing, and statistical analysis

**The goal** is to construct a dataframe in Python, perform a cursory inspection of the provided dataset, and inform TikTok data team members of your findings.

*This activity has three parts:*
- **Part 1:** Understand the situation
- **Part 2:** Understand the data
- **Part 3:** Understand the variables

---
# PACE: Plan

## Task 1. Understand the situation

**How can you best prepare to understand and organize the provided TikTok information?**

**Response:** Prepare by reading in the data, reviewing the Data Dictionary provided by the team, and exploring the dataset to identify key variables for the stakeholder. The Data Dictionary tells us we have 19,383 rows (one per TikTok video) and 12 columns covering claim status, video metadata, author status, and engagement metrics. The primary goal is to understand the structure before any modeling begins.

---
# PACE: Analyze

## Task 2a. Imports and data loading

In [ ]:
# Import packages
import pandas as pd
import numpy as np

In [ ]:
# Load the dataset into a dataframe
data = pd.read_csv('tiktok_dataset.csv')

## Task 2b. Understand the data — Inspect the data

In [ ]:
# Display and examine the first 10 rows of the dataframe
data.head(10)

**Question 1:** When reviewing the first few rows of the dataframe, what do you observe about the data? What does each row represent?

**Response:** Each row represents a unique TikTok video that has been flagged as containing either a claim or an opinion. The first 10 rows are all labeled as 'claim' with 'not verified' author status. The engagement columns (view count, like count, etc.) contain float values, suggesting some rows may have null values. The `video_transcription_text` column contains the actual spoken content of the video, which will be important for the eventual NLP-based classification model.

In [ ]:
# Get summary info about the dataframe
data.info()

**Question 2:** When reviewing the `data.info()` output, what do you notice about the different variables? Are there any null values? Are all of the variables numeric? Does anything else stand out?

**Response:** Several observations stand out:
- The dataset has **19,383 rows and 12 columns**
- **5 columns have 298 null values each**: `video_view_count`, `video_like_count`, `video_share_count`, `video_download_count`, and `video_comment_count` — all engagement-related columns
- **Not all columns are numeric**: `claim_status`, `video_transcription_text`, `verified_status`, and `author_ban_status` are object (string) type
- The engagement columns are stored as **float64** rather than int64, which is why they can accommodate null values
- `video_id` is stored as int64 but functions as an identifier, not a numeric measure

In [ ]:
# Get a summary of descriptive statistics for the dataframe
data.describe()

**Question 3:** When reviewing the `data.describe()` output, what do you notice about the distributions of each variable? Are there any questionable values? Does it seem that there are outlier values?

**Response:** Several observations:
- **Video duration** ranges from 5 to 60 seconds, which is consistent with TikTok's short-form video format — no obvious outliers here
- **Engagement counts show extreme right skew**: For example, `video_view_count` has a mean much higher than the 25th percentile, suggesting many videos with low views and a small number of videos with extremely high views — classic long-tail distribution
- **The maximum values for engagement metrics are very high** — this is expected on a major platform but these extreme values will need to be considered during analysis
- The `#` column (row index) runs sequentially from 1 to 19,383, confirming no duplicate rows

---
## Task 3. Understand the variables

Use insights from your examination of the summary data to guide deeper investigation into variables.

In [ ]:
# What are the different values for claim_status?
print(data['claim_status'].value_counts())
print(f"\nPercentage breakdown:")
print(data['claim_status'].value_counts(normalize=True).round(4) * 100)

In [ ]:
# What are the different values for author_ban_status?
print(data['author_ban_status'].value_counts())
print(f"\nPercentage breakdown:")
print(data['author_ban_status'].value_counts(normalize=True).round(4) * 100)

In [ ]:
# Examine min and max values for video_duration_sec
print(f"video_duration_sec:")
print(f"  Min: {data['video_duration_sec'].min()} seconds")
print(f"  Max: {data['video_duration_sec'].max()} seconds")
print(f"  Mean: {data['video_duration_sec'].mean():.2f} seconds")

print(f"\nvideo_view_count:")
print(f"  Min: {data['video_view_count'].min():,.0f}")
print(f"  Max: {data['video_view_count'].max():,.0f}")
print(f"  Mean: {data['video_view_count'].mean():,.2f}")

**Observation:** `video_duration_sec` ranges from 5 to 60 seconds — consistent with TikTok's platform constraints. `video_view_count` ranges widely from very low to nearly 1 million, confirming the heavy right-skewed distribution noted in `data.describe()`.

In [ ]:
# Check the number of null values across columns
print("Null value counts per column:")
print(data.isnull().sum())
print(f"\nTotal null values: {data.isnull().sum().sum()}")
print(f"Percentage of rows with nulls: {(298/19383*100):.2f}%")

**Observation:** 298 rows (~1.54% of the dataset) have null values across all 5 engagement columns simultaneously. This suggests these rows may represent videos where engagement data was not recorded or was removed — likely a data collection issue rather than random missingness. These rows will need to be handled before modeling.

In [ ]:
# Examine engagement metrics grouped by author_ban_status
data.groupby(['author_ban_status']).agg(
    {'video_view_count': ['count', 'mean', 'median'],
     'video_like_count': ['count', 'mean', 'median'],
     'video_share_count': ['count', 'mean', 'median']})

**Observation:** Banned authors and those under review get far more views, likes, and shares than active authors. In most groups, the mean is much greater than the median, indicating some videos with very high engagement counts — a highly skewed distribution.

---
## Task 4. Create new engagement rate variables

Create three new columns to better understand engagement rates:
- `likes_per_view`: number of likes divided by number of views
- `comments_per_view`: number of comments divided by number of views  
- `shares_per_view`: number of shares divided by number of views

In [ ]:
# Create a likes_per_view column
data['likes_per_view'] = data['video_like_count'] / data['video_view_count']

# Create a comments_per_view column
data['comments_per_view'] = data['video_comment_count'] / data['video_view_count']

# Create a shares_per_view column
data['shares_per_view'] = data['video_share_count'] / data['video_view_count']

print("New columns created successfully.")
print("\nSample of new engagement rate columns:")
data[['claim_status', 'author_ban_status', 'likes_per_view', 
      'comments_per_view', 'shares_per_view']].head()

In [ ]:
# Group by claim_status and author_ban_status, calculate count/mean/median
data.groupby(['claim_status', 'author_ban_status']).agg(
    {'likes_per_view': ['count', 'mean', 'median'],
     'comments_per_view': ['count', 'mean', 'median'],
     'shares_per_view': ['count', 'mean', 'median']})

**Observation:** While banned authors and those under review get far more raw views, when a video *does* get viewed, engagement rate is more related to **claim status** than author ban status. Claim videos consistently have higher likes_per_view (~0.33) compared to opinion videos (~0.22), as well as higher shares_per_view and comments_per_view. This pattern holds across all three ban status categories — claim status is a stronger predictor of engagement rate than ban status.

---
# PACE: Execute

## Task 5. Summary for Rosie Mae Bradshaw and the TikTok data team

**Given your efforts, what can you summarize for the TikTok data team?**

**Response:**

**Dataset overview:**
- The dataset contains 19,383 rows and 12 columns. Each row represents a unique TikTok video flagged as containing either a claim or an opinion.
- Of the 19,383 samples, approximately **49.6% are claims** (~9,608 videos) and **50.4% are opinions** (~9,750 videos) — a near-balanced dataset, which is favorable for machine learning classification.

**Key findings:**
- **298 rows (~1.54%) have null values** across all 5 engagement columns. These will need to be addressed before modeling — options include dropping these rows or imputing values.
- **Engagement level is strongly correlated with claim status.** Claim videos consistently receive higher likes-per-view (~0.33) and shares-per-view (~0.065) than opinion videos (~0.22 and ~0.044 respectively). This engagement pattern could be a powerful predictive feature.
- **Banned authors generate significantly more raw views, likes, and shares** than active authors. However, when normalized by view count, claim status becomes the stronger driver of engagement rate — suggesting ban status and claim status are related but distinct signals.

**Recommended next steps:**
1. Handle the 298 null rows — investigate whether they are random or systematic before deciding to drop or impute
2. Conduct full EDA including visualizations of engagement distributions by claim status
3. Explore the `video_transcription_text` column — NLP features derived from the transcription text will likely be central to the classification model
4. Investigate the relationship between `author_ban_status` and `claim_status` more deeply — banned authors appear disproportionately associated with claims
5. Begin feature engineering for the machine learning pipeline